In [1]:
import pandas as pd
import numpy as np

# Load the labelled dataset
df = pd.read_csv("../data/processed/kerala_flood_labelled.csv")

# Sort chronologically
df = df.sort_values("YEAR").reset_index(drop=True)

# Recreate our chronological 70:30 split
split_index = round(len(df) * 0.70)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining flood cases:", train_df["FLOOD"].sum())
print("Testing flood cases:", test_df["FLOOD"].sum())

Training shape: (82, 20)
Testing shape: (35, 20)

Training flood cases: 3
Testing flood cases: 3


In [2]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Separate input features and target
feature_columns = ["JAN", "FEB", "MAR", "APR", "MAY"]

X_train = train_df[feature_columns]
y_train = train_df["FLOOD"]

X_test = test_df[feature_columns]
y_test = test_df["FLOOD"]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training features: (82, 5)
Testing features: (35, 5)
Training labels: (82,)
Testing labels: (35,)


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create Logistic Regression pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

print(lr_pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])


In [4]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# Define 3-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# Perform cross-validation
lr_cv_results = cross_validate(
    lr_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

# Display results
for metric in scoring:
    scores = lr_cv_results[f"test_{metric}"]

    print(f"\n{metric.upper()}")
    print("Fold scores:", scores)
    print("Mean score:", scores.mean())


RECALL
Fold scores: [0. 0. 1.]
Mean score: 0.3333333333333333

F1
Fold scores: [0.         0.         0.16666667]
Mean score: 0.05555555555555555

ROC_AUC
Fold scores: [0.07407407 0.34615385 0.80769231]
Mean score: 0.409306742640076


In [5]:
from sklearn.metrics import (
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Train on the complete training dataset
lr_pipeline.fit(X_train, y_train)

# Generate predictions
y_pred = lr_pipeline.predict(X_test)

# Get flood probabilities for ROC-AUC
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
print("Logistic Regression - Test Results")

print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Logistic Regression - Test Results
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.3229166666666667

Confusion Matrix:
[[23  9]
 [ 3  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.72      0.79        32
           1       0.00      0.00      0.00         3

    accuracy                           0.66        35
   macro avg       0.44      0.36      0.40        35
weighted avg       0.81      0.66      0.73        35



In [6]:
lr_test_results = test_df[["YEAR", "FLOOD"]].copy()

lr_test_results["PREDICTED_FLOOD"] = y_pred
lr_test_results["FLOOD_PROBABILITY"] = y_prob.round(3)

print(
    lr_test_results[lr_test_results["FLOOD"] == 1]
)

     YEAR  FLOOD  PREDICTED_FLOOD  FLOOD_PROBABILITY
91   1992      1                0              0.119
106  2007      1                0              0.155
112  2013      1                0              0.206


In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

# Create Decision Tree pipeline
dt_pipeline = Pipeline([
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=2,
        min_samples_leaf=5,
        random_state=42
    ))
])

print(dt_pipeline)

Pipeline(steps=[('classifier',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=2,
                                        min_samples_leaf=5, random_state=42))])


In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# Three-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# Perform cross-validation
dt_cv_results = cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

# Display results
for metric in scoring:
    scores = dt_cv_results[f"test_{metric}"]

    print(f"\n{metric.upper()}")
    print("Fold scores:", scores)
    print("Mean score:", scores.mean())


RECALL
Fold scores: [0. 0. 0.]
Mean score: 0.0

F1
Fold scores: [0. 0. 0.]
Mean score: 0.0

ROC_AUC
Fold scores: [0.42592593 0.46153846 0.42307692]
Mean score: 0.4368471035137702


In [9]:
from sklearn.metrics import (
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Train on the complete training dataset
dt_pipeline.fit(X_train, y_train)

# Generate predictions
dt_pred = dt_pipeline.predict(X_test)

# Calculate flood probabilities
dt_prob = dt_pipeline.predict_proba(X_test)[:, 1]

# Evaluate the model
print("Decision Tree - Test Results")

print("Recall:", recall_score(y_test, dt_pred, zero_division=0))
print("F1-score:", f1_score(y_test, dt_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, dt_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, dt_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y_test, dt_pred, zero_division=0))

Decision Tree - Test Results
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.359375

Confusion Matrix:
[[23  9]
 [ 3  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.72      0.79        32
           1       0.00      0.00      0.00         3

    accuracy                           0.66        35
   macro avg       0.44      0.36      0.40        35
weighted avg       0.81      0.66      0.73        35



In [10]:
dt_test_results = test_df[["YEAR", "FLOOD"]].copy()

dt_test_results["PREDICTED_FLOOD"] = dt_pred
dt_test_results["FLOOD_PROBABILITY"] = dt_prob.round(3)

print(dt_test_results[dt_test_results["FLOOD"] == 1])

     YEAR  FLOOD  PREDICTED_FLOOD  FLOOD_PROBABILITY
91   1992      1                0                0.0
106  2007      1                0                0.0
112  2013      1                0                0.0
